<a id="top"></a>

# Lab L1203: Read traces and locate failures

```yaml
title: "Lab L1203: Read traces and locate failures"
keywords: tracing, trace, span, narrate, runaway loop, wrong arguments, failure signature, lab
estimated duration: 30
```

> **Lesson:** L12. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L12/objectives.md).
> Runs offline — no API key needed (scripted `FakeModel`).

**Learning objectives.** By the end of this lab you can:

- **read a trace and narrate a run** — walk the spans top to bottom and say what the agent did;
- **locate and name a failure signature from the trace alone** — without re-running the agent.

> Solutions: `L1203_lab_solutions.ipynb`.
> Pure Python — no API key needed.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Narrate the good run](#2-problem-1--narrate-the-good-run)
- [3. Problem 2 — Find the runaway](#3-problem-2--find-the-runaway)
- [4. Problem 3 — Spot the wrong argument](#4-problem-3--spot-the-wrong-argument)
- [5. Problem 4 — Classify the signatures](#5-problem-4--classify-the-signatures)
- [6. Self-check](#6-self-check)

## 1. Setup

Run this cell once. It imports the frozen `common/` library and builds **five traces** from scripted `FakeModel` runs — the same loop you traced in lecture, offline and deterministic (no API key, no network). You won't read the scripts again; you'll read the **traces** they produced.

In [1]:
from fluffy_potato_curriculum.common.agent_loop import RunResult, run
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS

# good -- a clean multi-step run that terminates naturally.
good = run(
    FakeModel(
        [
            tool_reply(tool_call("c1", "calculator", {"expression": "17*23"})),
            tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),
            text_reply("17 * 23 is 391, and Tokyo has about 37,000,000 people."),
        ]
    ),
    TOOLS,
    "What is 17 * 23, and what is the population of Tokyo?",
)

# tool_error -- a tool raises; the loop turns it into an error ToolMessage
# (status="error"); model gives up.
tool_error = run(
    FakeModel(
        [
            tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
            text_reply("I couldn't fetch that URL -- it errored."),
        ]
    ),
    TOOLS,
    "Fetch https://crash and tell me what it says.",
)

# wrong_args -- the call SUCCEEDS, but the model looked up the wrong city.
wrong_args = run(
    FakeModel(
        [
            tool_reply(tool_call("c1", "lookup", {"city": "Paris"})),
            text_reply("Tokyo has about 11,000,000 people."),  # wrong: that's Paris
        ]
    ),
    TOOLS,
    "What is the population of Tokyo?",
)

# runaway -- the same failing call repeats forever; max_steps forces a halt.
runaway = run(
    FakeModel(
        [
            tool_reply(tool_call("c1", "lookup", {"city": "Atlantis"})),  # FakeModel repeats this
        ]
    ),
    TOOLS,
    "What is the population of Atlantis?",
    max_steps=4,
)

# premature -- the model answers with no tool call at all; the answer is a guess.
premature = run(
    FakeModel(
        [
            text_reply("It's probably around 5 million."),  # never looked it up
        ]
    ),
    TOOLS,
    "What is the population of Tokyo?",
)

traces: dict[str, RunResult] = {
    "good": good,
    "tool_error": tool_error,
    "wrong_args": wrong_args,
    "runaway": runaway,
    "premature": premature,
}
print("built traces:", ", ".join(traces))

built traces: good, tool_error, wrong_args, runaway, premature


[↑ Back to top](#top)

## 2. Problem 1 — Narrate the good run

Read a trace top to bottom. Fill in the loop body so it prints `span.one_line()` for each span in `good.trace`, in order. Then run it — the printed lines *are* the run, narrated.

The `assert` at the end checks you printed one line per span.

In [2]:
lines: list[str] = []
for span in good.trace:
    line = span.one_line()
    lines.append(line)
    print(line)

assert len(lines) == len(good.trace)

chain agent_loop.run       {'termination': 'natural', 'iterations': 3}
llm   model.invoke         -> tool_calls=['calculator'] tokens=120
tool  calculator           {'expression': '17*23'} -> '391'
llm   model.invoke         -> tool_calls=['lookup'] tokens=120
tool  lookup               {'city': 'Tokyo'} -> '37000000'
llm   model.invoke         -> tool_calls=[] tokens=120


**TODO (written):** Which span is the *natural-termination point* — the one that tells you the model decided it was done? Edit this cell with your answer.

_Answer:_ The **last `llm` span** (`[5] llm model.invoke -> tool_calls=[]`). Its `tool_calls` list is empty: the model emitted only text and asked for no tool, so the loop returned its text and stopped with `termination == "natural"`. (The final `chain`-span summary records that outcome, but the *decision* lives in that last `llm` span.)

[↑ Back to top](#top)

## 3. Problem 2 — Find the runaway

The runaway signature is **the same `(name, inputs)` repeating across tool spans, ending in `max_steps`**. Make `inputs` hashable so you can count repeats with `tuple(sorted(span.inputs.items()))`.

Fill in `most_repeated_call(result)` to return the `(name, inputs_tuple)` pair that occurs most often across the run's `tool` spans, paired with its count. Then the asserts check that the `runaway` trace repeats one call and ended in `max_steps`.

In [3]:
from collections import Counter


def most_repeated_call(result: RunResult) -> tuple[tuple[str, tuple[tuple[str, object], ...]], int]:
    """Return ((tool_name, sorted_inputs_tuple), count) for the most-repeated tool call."""
    calls = [
        (span.name, tuple(sorted(span.inputs.items())))
        for span in result.trace
        if span.run_type == "tool"
    ]
    (call, count) = Counter(calls).most_common(1)[0]
    return (call, count)


call, count = most_repeated_call(runaway)
print("most repeated:", call, "x", count)
print("termination:", runaway.termination)

# Runaway signature: one call repeated, and the run hit the step cap instead of stopping naturally.
assert count > 1
assert runaway.termination == "max_steps"

most repeated: ('lookup', (('city', 'Atlantis'),)) x 4
termination: max_steps


[↑ Back to top](#top)

## 4. Problem 3 — Spot the wrong argument

The `wrong_args` run terminates `natural` with **no error set** — yet the answer is wrong. The task asked about **Tokyo**, but the model looked up the wrong city. The bug is only visible in the `inputs`.

Pull the `lookup` tool span out of `wrong_args.trace`, read its `inputs["city"]`, and assert the looked-up city was **not** `"Tokyo"` (that's the bug). Then answer the written TODO.

In [4]:
lookup_span = next(span for span in wrong_args.trace if span.name == "lookup")
looked_up_city = lookup_span.inputs["city"]

print("termination:", wrong_args.termination, "| error:", lookup_span.error)
print("looked up city:", looked_up_city, "(task asked about Tokyo)")

# The bug: the model looked up some city other than the one the task asked for.
assert looked_up_city != "Tokyo"

termination: natural | error: None
looked up city: Paris (task asked about Tokyo)


**TODO (written):** A "did the tool call succeed?" flag would report this run as **green** — the call ran and returned a valid population. Why doesn't a success flag catch this bug, and what *does*? Edit this cell with your answer.

_Answer:_ The call genuinely **succeeded** — `lookup("Paris")` is a valid call that returns a real number — so any success/`is_error` flag is `False` and the run looks clean. The failure isn't in *whether* the tool ran; it's in *what it was asked*. Only reading the **arguments** (`inputs["city"] == "Paris"` when the task asked about Tokyo) reveals it. That's why tracing arguments — not just call names or success flags — is the skill: a tool can answer the wrong question perfectly.

[↑ Back to top](#top)

## 5. Problem 4 — Classify the signatures

You've now read four failing/clean traces. Tie it together: map each trace to its signature name and the **one field you'd read** to spot it. The four signature names are **tool error**, **wrong arguments**, **runaway loop**, **premature termination** (and `good` is the clean baseline).

Run the cell below to reprint each trace, then fill in the table in the next markdown cell.

In [5]:
# Reprint every trace so you can compare signatures side by side (no edits needed here).
for label, result in traces.items():
    print(f"=== {label} (termination={result.termination}) ===")
    for span in result.trace:
        print("  ", span.one_line())
    print()

=== good (termination=natural) ===
   chain agent_loop.run       {'termination': 'natural', 'iterations': 3}
   llm   model.invoke         -> tool_calls=['calculator'] tokens=120
   tool  calculator           {'expression': '17*23'} -> '391'
   llm   model.invoke         -> tool_calls=['lookup'] tokens=120
   tool  lookup               {'city': 'Tokyo'} -> '37000000'
   llm   model.invoke         -> tool_calls=[] tokens=120

=== tool_error (termination=natural) ===
   chain agent_loop.run       {'termination': 'natural', 'iterations': 2}
   llm   model.invoke         -> tool_calls=['flaky_fetch'] tokens=120
   tool  flaky_fetch          {'url': 'https://crash'} -> "RuntimeError('connection reset by peer')"  [is_error]
   llm   model.invoke         -> tool_calls=[] tokens=120

=== wrong_args (termination=natural) ===
   chain agent_loop.run       {'termination': 'natural', 'iterations': 2}
   llm   model.invoke         -> tool_calls=['lookup'] tokens=120
   tool  lookup               {'

**TODO (written):** Fill the **Signature** and **Tell (field to read)** columns. Edit this cell.

| Trace | Signature | Tell (field to read) |
| --- | --- | --- |
| `tool_error` | tool error | a `tool` span's `error` is set (`[is_error]`) |
| `wrong_args` | wrong arguments | a `tool` span's `inputs` answer the wrong question (no error, `natural`) |
| `runaway` | runaway loop | same `(name, inputs)` repeats; `termination == "max_steps"` |
| `premature` | premature termination | `natural` stop with **zero** `tool` spans; answer is a guess |
| `good` | (none — clean baseline) | `natural` stop, tools called with correct `inputs`, no errors |

[↑ Back to top](#top)

## 6. Self-check

Compare against `L1203_lab_solutions.ipynb`. Done when:

- **Problem 1** prints one `.one_line()` per span and its `assert` passes; your written answer names the **last `llm` span** (empty `tool_calls`) as the natural-termination point.
- **Problem 2** `most_repeated_call(runaway)` returns the repeated `lookup` call with count > 1, and both asserts pass.
- **Problem 3** reads `inputs["city"]` from the `lookup` span, the `!= "Tokyo"` assert passes, and your written answer explains why a success flag misses a wrong-argument bug.
- **Problem 4** the table maps all four traces to their signature names and the field that reveals each.

You can now read a trace, narrate a run, and name a failure from the record alone — no re-running. In **L13** these four signatures become your first eval cases.

[↑ Back to top](#top)